## ECommerce Returns Cleaning Phase

- BSIT 3A-G1 Group 1
- Adriano, Arianne Denisse
- Cabataña, Jocky King
- Elopre, Ross Anthony
- Pelayo, Danielle Griz
- Zaldivar, Jaen Royce

In [1]:
%pip install pandas numpy matplotlib seaborn scikit-learn

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.5 MB 230.2 kB/s eta 0:00:40
   -- ------------------------------------- 0.5/9.5 MB 230.2 kB/s eta 0:00:40
   -- ------------------------------------- 0.5/9.5 MB 230.2 kB/s eta 0:00:40
   --- 


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
# Imports
import pandas as pd
import numpy as np

In [3]:
dataset_path = './datasets/'
raw_folder_path = dataset_path + '/raw/'
file_name = 'ecommerce_returns_dirty.csv'

df = pd.read_csv(raw_folder_path + file_name)
df_clean = df.copy()
print("Before:", df_clean.shape, "| nulls:", df_clean.isna().sum().sum())

Before: (10000, 13) | nulls: 1194


In [ ]:
# Check variables
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Order ID           10000 non-null  str    
 1   Date               10000 non-null  str    
 2   Time of Purchase   10000 non-null  str    
 3   Customer Age       10000 non-null  int64  
 4   Customer Gender    10000 non-null  str    
 5   Device Used        10000 non-null  str    
 6   Product Category   10000 non-null  str    
 7   Total Order Value  9606 non-null   float64
 8   Discount Applied   10000 non-null  float64
 9   Payment Method     10000 non-null  str    
 10  Shipping Type      10000 non-null  str    
 11  Customer Rating    9200 non-null   float64
 12  Return Status      10000 non-null  str    
dtypes: float64(3), int64(1), str(9)
memory usage: 1015.8 KB


In [5]:
# Check for descriptive statistics
df_clean.describe()

,Customer Age,Total Order Value,Discount Applied,Customer Rating
count,10000.000000,9606.000000,10000.000000,9200.000000
mean,44.538800,12513.504313,0.232721,3.354457
std,20.359385,7131.774577,0.172686,1.367280
min,-12.000000,-996.600000,-0.500000,1.000000
25%,31.000000,6282.620000,0.110000,2.000000
50%,43.000000,12595.275000,0.240000,4.000000
75%,57.000000,18545.032500,0.370000,4.000000
max,200.000000,24994.840000,0.500000,5.000000


In [6]:
# Check for initial rows
df_clean.head(20)

,Order ID,Date,Time of Purchase,Customer Age,Customer Gender,Device Used,Product Category,Total Order Value,Discount Applied,Payment Method,Shipping Type,Customer Rating,Return Status
0,ORD-100000,2024-10-16,Morning,56,Female,Mobile,Sports,4144.83,0.42,Credit Card,Standard,2.0,Kept
1,ORD-100001,2023-04-25,Morning,69,Male,Mobile,CLOTHING,21436.04,0.01,E-Wallet,Standard,3.0,Returned
2,ORD-100002,2023-01-26,Midnight,46,Male,Mobile,Home,15655.16,0.10,Cash on Delivery,Express,NaN,Returned
3,ORD-100003,2025-01-29,Evening,32,female,Desktop,Electronics,NaN,0.44,E-Wallet,Standard,3.0,Kept
4,ORD-100004,2023-10-09,Afternoon,60,Male,Mobile,Home,23072.84,-0.49,E-Wallet,Standard,4.0,Kept
5,ORD-100005,2023-09-08,Afternoon,25,Male,Desktop,Beauty,6596.37,0.11,Cash on Delivery,Standard,5.0,Kept
6,ORD-100006,2023-08-17,Afternoon,38,Female,Desktop,Beauty,6369.90,0.18,Debit Card,Standard,5.0,Kept
7,ORD-100007,2023-05-23,Evening,56,Male,Desktop,Beauty,13073.24,0.38,Credit Card,Standard,2.0,Returned
8,ORD-100008,2025-01-24,Morning,36,Prefer not to say,Mobile,Electronics,18896.15,0.28,Cash on Delivery,Standard,5.0,Returned
9,ORD-100009,2023-04-15,Morning,40,Female,Desktop,SPORTS,11457.40,0.48,Cash on Delivery,Express,4.0,Kept


In [35]:
# Inconsistent Text
for col in ['Customer Gender', 'Product Category']:
    df_clean[col] = df_clean[col].str.strip().str.title()

df_clean['Customer Gender'] = df_clean['Customer Gender'].replace({'M': 'Male', 'F': 'Female'})
print(df_clean['Customer Gender'].value_counts())
print(df_clean['Product Category'].value_counts())

Customer Gender
Male                 4554
Female               4467
Prefer Not To Say     979
Name: count, dtype: int64
Product Category
Clothing       3493
Electronics    2384
Home           1606
Beauty         1470
Sports         1047
Name: count, dtype: int64


In [36]:
# Date Format Fix
def convert_date(s):
    if '/' in s:
        return pd.to_datetime(s, format='%m/%d/%Y', errors='coerce')
    if s[2] == '-':
        return pd.to_datetime(s, format='%d-%m-%Y', errors='coerce')
    return pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')

df_clean['Date'] = df_clean['Date'].astype(str).str.strip().apply(convert_date)
print('Uncleaned dates:', df_clean['Date'].isna().sum())

Uncleaned dates: 0


In [37]:
# Extreme Values to NaN
df_clean.loc[~df_clean['Customer Age'].between(18, 100), 'Customer Age'] = np.nan
df_clean.loc[df_clean['Total Order Value'] <= 0, 'Total Order Value'] = np.nan
df_clean['Discount Applied'] = df_clean['Discount Applied'].abs()

In [38]:
# Missing Values
df_clean['Rating Missing'] = df_clean['Customer Rating'].isna().astype(int)

df_clean['Customer Rating'] = df_clean['Customer Rating'].fillna(df_clean['Customer Rating'].median()) # median imputation since median is not affected by extreme values
df_clean['Customer Age'] = df_clean['Customer Age'].fillna(df_clean['Customer Age'].median())
df_clean['Total Order Value'] = df_clean.groupby('Product Category')['Total Order Value'].transform(lambda x: x.fillna(x.median()))

In [39]:
# Outlier Check and Verification
for col in ['Customer Age', 'Total Order Value']:
    q1, q3 = df_clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n = ((df_clean[col] < q1 - 1.5 * iqr) | (df_clean[col] > q3 + 1.5 * iqr)).sum()
    print(col, "IQR outliers:", n)

print("After:", df_clean.shape, "| nulls:", df_clean.isna().sum().sum())
df_clean.describe().T

Customer Age IQR outliers: 0
Total Order Value IQR outliers: 0
After: (10000, 14) | nulls: 0


,count,mean,min,25%,50%,75%,max,std
Date,10000,2024-05-16 15:54:51.840000,2023-01-01 00:00:00,2023-09-14 00:00:00,2024-05-20 12:00:00,2025-01-15 00:00:00,2025-09-27 00:00:00,NaN
Customer Age,10000.0,43.5162,18.0,31.0,43.0,56.0,69.0,14.753105
Total Order Value,10000.0,12654.593928,500.14,6863.3775,12708.48,18325.9275,24994.84,6861.428204
Discount Applied,10000.0,0.251241,0.0,0.13,0.25,0.38,0.5,0.144418
Customer Rating,10000.0,3.4061,1.0,3.0,4.0,4.0,5.0,1.323086
Rating Missing,10000.0,0.08,0.0,0.0,0.0,0.0,1.0,0.271307


In [ ]:
df_clean.shape[0]